In [ ]:
from pathlib import Path
from struct import unpack
from itertools import batched
from IORlib.ECL import INIT_file, UNRST_file
from IORlib.utils import flatten
import numpy as np

barrel2metric = 1.0 / 6.289811 # 0.158987294928


def read_ijk(root):
    filename = root.parent/'IJK.bin'
    with open(filename, 'rb') as file:
        size = unpack('L', file.read(8))[0]
        return unpack(f'{size}L', file.read(8*size))

def porevol_reader(root):
    files = sorted(root.parent.glob('porevol_???.bin'))
    for filename in files:
        print(filename)
        with open(filename, 'rb') as file:
            size = unpack('L', file.read(8))[0]
            yield unpack(f'{size}d', file.read(8*size))

shared = Path('/home/shared/owner')
#root = shared/'CoP_files/output/IORSIM_BASECASE_1971TO2022/IORSIM_BASECASE_1971TO2022'
root = shared/'CoP_files/input/IORSIM_BASECASE/IORSIM_BASECASE_INPUT'

unrst = UNRST_file(root)
ior_porv = np.zeros(unrst.dim())

ijk = [(a-1, b-1, c-1) for a,b,c in batched(read_ijk(root), 3)]
A = np.arange(len(ijk))

i = 0
for ior, ix in zip(porevol_reader(root), unrst.cellarray('RPORV', step=1)):
    for n in range(len(ijk)):
        ior_porv[ijk[n]] = ior[n]
    ix_porv = ix.rporv[0] * barrel2metric
    print(ior_porv[141, 92, 9], ix_porv[141, 92, 9])
    diff = ix_porv - ior_porv
    if abs(np.sum(diff)) > 0:
        print(f'{i} RPORVs differ: {diff[0,0,0]}')
    i += 1


#### Check that NNC block numbers and rates are the same in IORSim and UNRST

In [ ]:
from pathlib import Path
from struct import unpack
from itertools import batched
from IORlib.ECL import INIT_file, UNRST_file
from IORlib.utils import flatten
import numpy as np

barrel2metric = 0.158987294928/(24*3600)

def read_connections(root):
    filename = root.parent/'grid_connections.bin'
    with open(filename, 'rb') as file:
        size = 5 * unpack('L', file.read(8))[0]
        return unpack(f'{size}L', file.read(8*size))

def flow_reader(root):
    files = sorted(root.parent.glob('grid_flows_???.bin'))
    for filename in files:
        print(filename)
        with open(filename, 'rb') as file:
            size = 3 * unpack('L', file.read(8))[0]
            yield unpack(f'{size}d', file.read(8*size))

shared = Path('/home/shared/owner')
#root = shared/'CoP_files/output/IORSIM_BASECASE_1971TO2022/IORSIM_BASECASE_1971TO2022'
root = shared/'CoP_files/input/IORSIM_BASECASE_ITER2/IORSIM_BASECASE_INPUT'

conn = read_connections(root)
# Get NNC connections from IORSim-file
nnc_ior = [c[2:5] for c in batched(conn, 5) if c[1]==4]
# Get NNC block-numbers from the INIT-file
nnc_init = list(zip(*next(INIT_file(root).blockdata('NNC1', 'NNC2'))))
# Check if NNC1 and NNC2 are the same
nnc_diff = ((a[0]-b[0]-1, a[1]-b[1]-1) for a,b in zip(nnc_init, nnc_ior))
if sum(flatten(nnc_diff)) > 0:
    print('ERROR! NNC blocks differ!')

for flow, unrst in zip(flow_reader(root), UNRST_file(root).blockdata('FLRWATN+', 'FLROILN+', 'FLRGASN+')):
    ior_flows = np.array([f for c,f in zip(batched(conn, 5), batched(flow, 3)) if c[1]==4])
    ix_flows = np.array(list(zip(*unrst)))*barrel2metric
    diff = np.sum(ior_flows - ix_flows, axis=0)
    print(diff)

#### Check that NNC rates are the same in IORSim and UNRST

In [ ]:
from collections import namedtuple
from pathlib import Path
from ior2ecl import Output
from struct import unpack
from numpy import array, where
from itertools import batched

barrel2metric = 0.158987294928/(24*3600)

def read_flowfiles(*filenames):
    for name in filenames:
        with open(name, 'rb') as file:
            size = unpack('L', file.read(8))[0]
            yield unpack(f'{size}d', file.read(8*size))

def rates(root:Path):
    rates_ = namedtuple('Rates', 'gas oil wat')
    files = sorted(root.parent.glob('*rates_???.bin'), key=lambda x: (x.stem.split('_')[-1], x.name))
    for batch in batched(files, 3):
        yield rates_(*read_flowfiles(*batch))


shared = Path('/home/shared/owner')
#root = shared/'CoP_files/output/IORSIM_BASECASE_1971TO2022/IORSIM_BASECASE_1971TO2022'
root = shared/'CoP_files/input/IORSIM_BASECASE_ITER2/IORSIM_BASECASE_INPUT'
out = Output(root)
i = 0
for met, brl in zip(rates(root), out.slb_unrst.blockdata('FLRGASN+', 'FLROILN+', 'FLRWATN+')):
    print(i)
    for n in range(3):
        diff = where(array(brl[n])*barrel2metric - met[n] > 0)[0]
        if diff.size > 0:
            print(f'step {i}, phase {n}: {diff}')
    i += 1


#### Check that CONOTUB is the cumulative sum of CONORAT

In [47]:
# Compare CONORAT and CONOTUB
from itertools import islice
from IORlib.ECL import RFT_file
from IORlib.utils import pairwise
from pathlib import Path
shared = Path('/home/shared/owner')
#root = shared/'CoP_files/output/IORSIM_BASECASE_1971TO2022/IORSIM_BASECASE_1971TO2022'
root = shared/'CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM'
rft = RFT_file(root)
#rate, tube = next(rft.blockdata('CONORAT','CONOTUB', singleton=True))
rate, tube = list(rft.blockdata('CONORAT','CONOTUB', singleton=True))[201]
def show(iterable):
    print(', '.join(f'{i:.8f}' for i in iterable))
show(rate)
show([a-b for a,b in pairwise(tube)]+[tube[-1]])

0.00022899
0.00022899


### Compare reservoir and surface rates for blocks
Compare Intersect reservoir rates (FLROILI+, FLRWATI+, FLRGASI+) with reservoir   
rates calculated from Intersect surface rates (FLOOILI+, 'FLOWATI+, FLOGASI+)   
using formation factors (BO, BG, BW) and oil/gas ratios (RV, RS) 

In [3]:
from IORlib.ECL import UNRST_file
from pathlib import Path
import numpy as np
home = Path.home()
shared = Path('/home/shared/owner')
#root = shared/'CoP_files/input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'
root = shared/'CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM'
unrst = UNRST_file(root)
surf = ('FLOOILI+', 'FLOWATI+', 'FLOGASI+')
res =  ('FLROILI+', 'FLRWATI+', 'FLRGASI+')
conv = ('RS', 'RV', 'BO', 'BG', 'BW')
for ihead, *filemap in unrst.blockdata('INTEHEAD', *surf, *res, *conv):
    SO, SW, SG, RO, RW, RG, RS, RV, BO, BG, BW = [np.array(d) for d in filemap]

    # From IORSim code
    # rate.oil   = Bo * (     STC_rate.oil - Rv*STC_rate.gas )/( 1.0 - Rs*Rv );
    # rate.gas   = Bg * ( -Rs*STC_rate.oil +    STC_rate.gas )/( 1.0 - Rs*Rv );
    # rate.water = Bw *       STC_rate.water;

    reso = BO * (     SO - RV*SG ) / ( 1.0 - RS*RV )
    resg = BG * ( -RS*SO +    SG ) / ( 1.0 - RS*RV )
    resw = BW *       SW

    print(ihead[64:67])
    #N = (634722, 634933)
    N = range(20)
    for b in N:
        print('----------------------------------------')
        print(b, '   surf:', SO[b]  , SW[b]  , SG[b])
        print(b, '   read:', RO[b]  , RW[b]  , RG[b])
        print(b, '   calc:', reso[b], resw[b], resg[b])




(1, 1, 2000)
----------------------------------------
0    surf: 0.0 0.0 0.0
0    read: 0.0 0.0 0.0
0    calc: 0.0 0.0 0.0
----------------------------------------
1    surf: 0.0 0.0 0.0
1    read: 0.0 0.0 0.0
1    calc: 0.0 0.0 0.0
----------------------------------------
2    surf: 0.0 0.0 0.0
2    read: 0.0 0.0 0.0
2    calc: 0.0 0.0 0.0
----------------------------------------
3    surf: 0.0 0.0 0.0
3    read: 0.0 0.0 0.0
3    calc: 0.0 0.0 0.0
----------------------------------------
4    surf: 0.0 0.0 0.0
4    read: 0.0 0.0 0.0
4    calc: 0.0 0.0 0.0
----------------------------------------
5    surf: 0.0 0.0 0.0
5    read: 0.0 0.0 0.0
5    calc: 0.0 0.0 0.0
----------------------------------------
6    surf: 0.0 0.0 0.0
6    read: 0.0 0.0 0.0
6    calc: 0.0 0.0 0.0
----------------------------------------
7    surf: 0.0 0.0 0.0
7    read: 0.0 0.0 0.0
7    calc: 0.0 0.0 0.0
----------------------------------------
8    surf: 0.0 0.0 0.0
8    read: 0.0 0.0 0.0
8    calc: 0.0 0.0 0

### Compare reservoir and surface rates for wells
Compare Intersect well tubing rates at reservoir conditions (CONWTUBL, CONOTUBL, CONGTUBL)   
with reservoir rates calculated from Intersect well rates at surface conditions   
(CONWRAT, CONORAT,CONGRAT,) using formation factors (BO, BG, BW) and oil/gas ratios (RV, RS).  

The manual description of CONOTUB is 
>Oil flow rate through the tubing at the start of the connection (+ve towards wellhead),  
at surface conditions (that is CONORAT sum over this and all upstream connections, for producers)   

For Intersect runs, CONOTUBL is CONOTUB at reservoir conditions. The flow rates for each connection   
is then obtained by taking the difference CONORRAT = CONOTUBL[$i$] - CONOTUBL[$i+1$] 

In [5]:
from IORlib.ECL import UNRST_file, RFT_file, unfmt_file, SMSPEC_file
from pathlib import Path
import numpy as np
from collections import namedtuple
home = Path.home()
shared = Path('/home/shared/owner')
#root = shared/'CoP_files/input/IORSIM_BASECASE/IORSIM_BASECASE_INPUT'
root = shared/'CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM'
init = unfmt_file(root.with_suffix('.INIT'))
unrst = UNRST_file(root)
rft = RFT_file(root)

np.set_printoptions(3)

names = ('RS', 'RV', 'BO', 'BG', 'BW')
rst = namedtuple('rst', names)
rst_data = {str(day):rst(*keys) for day, keys in zip(unrst.days(), unrst.blockdata(*names))}
nx, ny, nz = next(init.blockdata('INTEHEAD'))[8:11]

def block(i,j,k):
    return i-1 + nx*( (j-1) + ny*(k-1) ) 

pos  = ('CONIPOS',  'CONJPOS',  'CONKPOS')
res  = ('CONWTUBL', 'CONOTUBL', 'CONGTUBL')
surf = ('CONWTUB',  'CONOTUB',  'CONGTUB')
orig = ('CONWRAT',  'CONORAT',  'CONGRAT')
for T, CX, WT, WP, I,J,K, *filemap in rft.blockdata('TIME', 'CONNXT', 'WELLETC', 'WELLPLT', *pos, *res, *surf, *orig, singleton=True):
    #   *res        *surf       *orig
    RTW,RTO,RTG, STW,STO,STG, SW,SO,SG = [np.array(d) for d in filemap]
    blocks = [block(*ijk) for ijk in zip(I,J,K)]
    time = str(int(T[0]))
    RS,RV, BO,BG,BW = [np.array(d)[blocks] for d in rst_data[time]]

    # rate.oil   = Bo * (     STC_rate.oil - Rv*STC_rate.gas )/( 1.0 - Rs*Rv );
    # rate.gas   = Bg * ( -Rs*STC_rate.oil +    STC_rate.gas )/( 1.0 - Rs*Rv );
    # rate.water = Bw *       STC_rate.water;
    
    reso = BO * (     SO - RV*SG ) / ( 1.0 - RS*RV )
    resg = BG * ( -RS*SO +    SG ) / ( 1.0 - RS*RV )
    resw = BW *       SW

    def diff(arr):
        return -np.diff(np.append(arr,0))
    dRTO = diff(RTO)
    dRTG = diff(RTG)
    dRTW = diff(RTW)
    dSTO = diff(STO)
    dSTW = diff(STW)

    def rel_err(V, ref):
        return (V-ref)
        #return 100*abs(V-ref)/(0.5*(ref+V))
    pO = rel_err(reso, dRTO)
    pG = rel_err(resg, dRTG)
    pW = rel_err(resw, dRTW)

    def show(val, err):
        return ', '.join((f'({a:.3f}, {b:.0f} %)' for a,b in zip(val, err)))
    
    #print(T[0], 'days, well:', WT[1]) #, ', O-rate:',WP[0], ', W-rate:',WP[1])
    #print('O', 'G', 'W')
    # print(reso, resg, resw)
    # print(dRTO, dRTG, dRTW)
    # print('W', dSTW, SW)
    #print(RTO, RTG, RTW)
    #print(STO, STG, STW)
    #print(SO, SG, SW)
    if 'Inj' in WT[1]:
        print(f'{WT[1]}: {RTW[0]:12.5e}, {STW[0]:12.5e}, {SW[0]:12.5e}')
    #print('W', dSTW, SW)
    # #print('   ', blocks)
    # reso0 = np.where(reso < 0)[0]
    # dRTO0 = np.where(dRTO < 0)[0]
    # if len(reso0) > 0:
    #     print('   reso < 0:', reso[reso0])
    # if len(dRTO0) > 0:
    #     print('   dRTO < 0:', dRTO0)
    #     #print('      CNNXT:', CX)
    #     print('   dRTO, reso:', list(enumerate(zip(dRTO, reso))))
    #     print('   dSTO, SO:', list(enumerate(zip(dSTO, SO))))
    #     #print('    RTO:', RTO)
    #     #print('       reso:', reso)
    # #print(' reso:', reso)
    # #print(' dRTO:', dRTO)
    #print('  pO:', show(reso, pO))
    #print('  pG:', show(resg, pG))
    #print('  pW:', show(resw, pW))





Inj_Comp: -9.94836e-09,  0.00000e+00, -9.05723e-04
Inj_Comp: -7.69543e-11,  0.00000e+00, -9.05733e-04
Inj_Comp: -2.43753e-11,  0.00000e+00, -9.05733e-04
Inj_Comp: -7.72366e-11,  0.00000e+00, -9.05733e-04
Inj_Comp: -8.86030e-11,  0.00000e+00, -9.05733e-04
Inj_Comp: -7.30531e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  8.38611e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  1.56980e-10,  0.00000e+00, -9.05733e-04
Inj_Comp:  8.03674e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  4.23733e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  2.86828e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  2.06560e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  1.63344e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  1.32881e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  1.13175e-11,  0.00000e+00, -9.05733e-04
Inj_Comp:  9.87483e-12,  0.00000e+00, -9.05733e-04
Inj_Comp:  8.54722e-12,  0.00000e+00, -9.05733e-04
Inj_Comp:  7.58554e-12,  0.00000e+00, -9.05733e-04
Inj_Comp:  6.82867e-12,  0.00000e+00, -9.05733e-04
Inj_Comp:  6.05475e-12,  0.0000

In [ ]:
for T, CX, WT, WP in rft.blockdata('TIME', 'CONNXT', 'WELLETC', 'WELLPLT'):
    print(WT)

In [ ]:
from IORlib.ECL import UNRST_file
from datetime import datetime
from pathlib import Path
home = Path.home()
shared = Path('/home/shared/owner')
root = shared/'CoP_files/input/IORSIM_BASECASE/IORSIM_BASECASE_INPUT'
root = shared/'norne/NORNE_INTERSECT'
root = home/'github/IORSim_GUI/cases/GEOCHEM_BACK/GEOCHEM_BACK'
root = home/'github/IORSim_GUI/cases/EKOFISK_HAUKAAS_2014_BACK/EKOFISK_HAUKAAS_2014_BACK'
#root = shared/'CoP_files/input/IORSIM_BASECASE_ITER3/IORSIM_BASECASE_INPUT'
#root = shared/'CoP_files/output/IORSIM_BASECASE_1971TO2022/IORSIM_BASECASE_1971TO2022'
#root = shared/'CoP_files/input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'
#out = Output(root)
#out.convert();
#out.merge();

unrst = UNRST_file(root)
#list(unrst.read2('sec'))
[datetime(*ymdhm) for ymdhm in unrst.read2('year', 'month', 'day', 'hour', 'min')]

